In [1]:
#Step 1
#Install mlxtend
#!pip install mlxtend

#Load libraries
import pandas as pd
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules



In [9]:
# STEP 2: Load and prepare Titanic data
df1 = sns.load_dataset('titanic')
print(f"original data: \n{df1.head()}\n")

df = df1[['sex', 'age', 'class', 'embarked', 'alone', 'survived']].dropna()
df['age_bin'] = pd.cut(df['age'], bins=[0, 12, 18, 35, 60, 100],
                       labels=['child', 'teen', 'young_adult', 'adult', 'senior'])
df['survived'] = df['survived'].map({0: 'died', 1: 'survived'})
df['alone'] = df['alone'].map({True: 'alone', False: 'not_alone'})
print(f"transformed data: \n{df.head()}")

original data: 
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  

transformed data: 
      sex   age  class embarked      alone  survived      age_bin
0    male  22.0  Third        S  not_alone      died  young_adult
1  female  38.0  First        C  not_alone  survive

Note: parch = Parent/Children abroad,  sibsp = Siblings/Spouse Aboard, deck = cabin deck letter, alone = traveling alone

In [10]:
#STEP 3: One-hot encode for Apriori
titanic_ohe = pd.get_dummies(df[['sex', 'class', 'embarked', 'alone', 'survived', 'age_bin']])
titanic_ohe.head()

,sex_female,sex_male,class_First,class_Second,class_Third,embarked_C,embarked_Q,embarked_S,alone_alone,alone_not_alone,survived_died,survived_survived,age_bin_child,age_bin_teen,age_bin_young_adult,age_bin_adult,age_bin_senior
0,False,True,False,False,True,False,False,True,False,True,True,False,False,False,True,False,False
1,True,False,True,False,False,True,False,False,False,True,False,True,False,False,False,True,False
2,True,False,False,False,True,False,False,True,True,False,False,True,False,False,True,False,False
3,True,False,True,False,False,False,False,True,False,True,False,True,False,False,True,False,False
4,False,True,False,False,True,False,False,True,True,False,True,False,False,False,True,False,False


In [11]:
#Step 4
# Run Apriori
frequent_itemsets = apriori(titanic_ohe, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

# Sort and show top rules
top_rules = rules.sort_values(by='lift', ascending=False).head(10)
top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
170,(age_bin_child),"(alone_not_alone, class_Third)",0.064607,0.666667,3.651282
206,"(embarked_C, sex_female)","(class_First, survived_survived)",0.051966,0.606557,3.598907
131,"(embarked_C, age_bin_adult)",(class_First),0.051966,0.902439,3.492047
422,"(sex_female, alone_not_alone, class_Second)","(embarked_S, survived_survived)",0.050562,0.818182,2.898236
217,"(sex_female, class_Second)","(embarked_S, survived_survived)",0.084270,0.810811,2.872126
218,"(class_Second, survived_survived)","(sex_female, embarked_S)",0.084270,0.722892,2.767198
212,"(sex_female, class_First)","(alone_not_alone, survived_survived)",0.071629,0.614458,2.734337
138,"(survived_survived, age_bin_adult)",(class_First),0.075843,0.701299,2.713721
421,"(alone_not_alone, class_Second, survived_survi...","(sex_female, embarked_S)",0.050562,0.705882,2.702087
204,"(embarked_C, sex_female, survived_survived)",(class_First),0.051966,0.672727,2.603162


In [36]:
#STEP 5: Strong rule mining
frequent_itemsets = apriori(titanic_ohe, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.8)

#Filter interesting rules
strong_rules = rules[rules['lift'] > 2.0].sort_values(by='lift', ascending=False)

#Display top rules
strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3)


,antecedents,consequents,support,confidence,lift
7,"(sex_female, class_First)",(survived_survived),0.11236,0.963855,2.382865


In [22]:
#STEP 6 Mining rules of interest

#frequent_itemsets = apriori(titanic_ohe, min_support=0.01, use_colnames=True)

#Force RHS to be of "died"
died_rules = rules[
    (rules['consequents'].apply(lambda x: 'survived_died' in x)) &
    (rules['consequents'].apply(lambda x: len(x) == 1))
]
died_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3)


,antecedents,consequents,support,confidence,lift
9,"(sex_male, class_Second)",(survived_died),0.117978,0.848485,1.424814
13,"(sex_male, class_Third)",(survived_died),0.301966,0.849802,1.427027
16,"(sex_male, embarked_S)",(survived_died),0.421348,0.815217,1.368950


In [26]:
#Mining rules of interest 2 (Force LHS to be of "Adult" and "3rd class")
adult_classC_rules = rules[
    rules['antecedents'].apply(lambda x: 'age_bin_young_adult' in x and 'class_Third' in x)
]

adult_classC_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3)


,antecedents,consequents,support,confidence,lift
32,"(age_bin_young_adult, class_Third)",(embarked_S),0.234551,0.843434,1.083981
47,"(sex_male, age_bin_young_adult, class_Third)",(embarked_S),0.182584,0.844156,1.084908
50,"(sex_male, age_bin_young_adult, class_Third)",(alone_alone),0.188202,0.870130,1.541126
